In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [2]:
q1 = "Can I still join the course after the start date?"
q2 = "Can I enroll late?"
q3 = "How do I install Docker?"

v1 = model.encode(q1)
v2 = model.encode(q2)
v3 = model.encode(q3)

In [3]:
print(v1.shape)
print(v1[:10])  # first 10 values

(384,)
[ 0.02139041 -0.07397997  0.00142069  0.02138166  0.02451131  0.03155828
 -0.1108397  -0.1050175  -0.06182589 -0.00642312]


In [10]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [5]:
v1.dot(dv)

np.float32(0.32332397)

In [6]:

v2 = model.encode(q3)

In [7]:
v2.dot(dv)

np.float32(0.07853731)

In [4]:
from ingest import load_faq_data

documents = load_faq_data()

In [5]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [6]:
from tqdm.auto import tqdm

In [7]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [8]:
import numpy as np
X = np.array(vectors)

In [9]:
X.shape

(1350, 384)

In [23]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [24]:
scores = X.dot(v_query)

In [25]:
scores = [v_query.dot(X[i]) for i in range(len(X))]

In [26]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [27]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [28]:
top5 = np.argsort(scores)[-5:]

In [29]:
top5 = top5[::-1]
top5

array([  2, 625, 907, 538,   7])

In [30]:
scores[top5]

TypeError: only integer scalar arrays can be converted to a scalar index

In [31]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.76294106
{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.", 'doc_id': '3f1424af17'}

0.7579372
{'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute.", 'doc_id': '2d8b16c2a0'}

0.7192131
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions'